# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will print a structured overview of all record sets and their fields, referencing by `@id` where possible.

In [ ]:
print("\nAvailable Record Sets:")
record_sets = list(metadata.record_sets)
for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet name: {rs.name}")
    print(f"    @id: {rs.id}")
    print(f"    Description: {rs.description}")
    print(f"    Fields:")
    for field in rs.fields:
        print(f"        - {field.name} (@id: {field.id}, type: {field.data_type})")
    print("")

# We'll display the first 2 records for each set, for initial exploration
for rs in record_sets:
    print(f"Sample records from RecordSet '{rs.name}' (@id: {rs.id}):")
    for i, rec in enumerate(dataset.records(record_set=rs.id)):
        print(rec)
        if i >= 1:
            break
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
All entities referenced below use their `@id` field.

In [ ]:
# ---
# Collect RecordSet IDs
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load full records into pandas DataFrame
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet with @id '{record_set_id}'. Columns:")
        print(dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for RecordSet with @id: {record_set_id}")
        dataframes[record_set_id] = pd.DataFrame([])


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
If the dataset contains a numeric field, e.g. 'MW' (molecular weight), show manipulation referencing by the field's `@id`.

In [ ]:
### Example on the main protein RecordSet
from IPython.display import display

# ---
# For demonstration, select the first RecordSet (often the main data table)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    raise RuntimeError('No record sets found!')

df = dataframes[main_record_set_id].copy()
print(f"Working with RecordSet @id: {main_record_set_id}")
display(df.head())

# Let's auto-detect a numeric field (e.g., MW, Coverage, Abundance, etc.) and a grouping field
import numpy as np
numeric_field = None
group_field = None

if not df.empty:
    # Detect numeric columns by dtype or typical field names
    for candidate in df.columns:
        try:
            df[candidate] = pd.to_numeric(df[candidate], errors="ignore")
        except Exception:
            pass
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Selected numeric field: {numeric_field}")
    else:
        print("No numeric field detected.")
    # Try to select a grouping field (e.g., category/description/sample/label)
    group_candidates = [col for col in df.columns if 'group' in col.lower() or 'sample' in col.lower() or 'condition' in col.lower()]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Selected grouping field: {group_field}")
    else:
        print("No obvious grouping field detected.")

    # EDA: filter, normalize, group
    if numeric_field is not None:
        threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else None
        if threshold is not None:
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records where '{numeric_field}' > {threshold:.2f}:")
            display(filtered_df.head())

            # Z-score normalization
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized '{numeric_field}' (z-score) for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Group by if possible
            if group_field in df.columns:
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by '{group_field}':")
                display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will create a histogram of the main numeric field per group, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    if group_field and group_field in df.columns:
        sns.histplot(data=df, x=numeric_field, hue=group_field, kde=True, element="step")
        plt.title(f"Distribution of '{numeric_field}' by '{group_field}'")
    else:
        sns.histplot(df[numeric_field], kde=True)
        plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we loaded mass spectrometry protein data from a Croissant schema, explored record sets and fields, filtered and normalized quantitative data, and visualized distributions. We referenced all entities by their Croissant `@id` in our scripts for reproducibility and schema compliance.

For more advanced analysis, see the [mlcroissant documentation](https://mlcommons.github.io/croissant-python/).